# DQN Pong — Prioritized Experience Replay (PER)
**Chapter 06 · Deep Reinforcement Learning Hands-On**

This notebook implements PER combined with NoisyNets (the best algorithm from Step 2).

**Baseline results for comparison:**
| Algorithm | Frames | Games | Mean Reward | vs Baseline |
|---|---|---|---|---|
| Epsilon-Greedy | 1,962,510 | 1,073 | 19.52 | baseline |
| Boltzmann | 1,457,724 | 687 | 19.55 | +25.7% |
| NoisyNets | 976,209 | 559 | 19.59 | +50.3% |

Enable **GPU T4** before running (Settings → Accelerator → GPU T4 x2).

## 1. Why Standard Replay Buffers Fall Short

### The problem: uniform sampling

The baseline DQN stores transitions `(s, a, r, s')` in a circular buffer and samples
them **uniformly at random** during training. Every experience has the same chance of
being replayed, regardless of how useful it is.

This creates two concrete problems:

**1. Wasted computation on already-learned transitions**  
Early in training the agent learns that moving the paddle *generally* works. After that,
the thousands of routine rally transitions are replayed over and over even though the
network no longer learns anything from them.

**2. Rare but critical experiences are under-sampled**  
A transition where the agent first succeeds at a difficult angle, or loses a point due
to a novel opponent strategy, may appear only a handful of times in the buffer.
Uniform sampling means these high-value transitions are drowned out by the majority.

### The solution: Prioritized Experience Replay (PER)

PER (Schaul et al., 2015) assigns each transition a **priority** based on its
**TD error** — how wrong the network's prediction was for that transition:

```
TD error δ = |Q(s,a) - (r + γ · max_a' Q_target(s', a'))|
```

A large TD error means the network was surprised — this transition contains useful
information. The sampling probability of each transition is:

```
P(i) = p_i^α / Σ p_k^α
```

where `α` controls how strongly we prioritize (0 = uniform, 1 = full prioritization).

### Bias correction with Importance Sampling (IS) weights

Non-uniform sampling introduces a bias — we are no longer estimating the true expected
gradient. To correct this, each transition's loss is scaled by an IS weight:

```
w_i = (1 / N · 1 / P(i))^β
```

`β` anneals from 0.4 → 1.0 over training. At β=1 the bias is fully corrected.
Early in training (β < 1) we allow some bias because the value estimates are too noisy
anyway — the prioritization benefit outweighs the small bias.

### Data structure: Sum Tree

Naively searching for the highest-priority transition is O(N). PER uses a **Sum Tree**
— a binary tree where each leaf is a priority and each internal node stores the sum of
its children. This gives O(log N) insertion, update, and sampling.

## 2. Install Dependencies

In [ ]:
!pip install -q 'gymnasium'
!pip install -q 'ale-py'
!pip install -q 'autorom[accept-rom-license]'
!AutoROM --accept-license --quiet
!pip install -q opencv-python-headless

import ale_py
import gymnasium as gym
gym.register_envs(ale_py)

env_test = gym.make('ALE/Pong-v5')
env_test.close()
print('ALE/Pong-v5 OK')

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Write Library Files

In [ ]:
import os
os.makedirs('lib', exist_ok=True)
open('lib/__init__.py', 'w').close()
print('lib/ ready')

In [ ]:
%%writefile lib/wrappers.py
import cv2
import ale_py
import gymnasium as gym
import gymnasium.spaces
import numpy as np
import collections

gym.register_envs(ale_py)


class FireResetEnv(gym.Wrapper):
    def __init__(self, env=None):
        super(FireResetEnv, self).__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == 'FIRE'
        assert len(env.unwrapped.get_action_meanings()) >= 3

    def step(self, action):
        return self.env.step(action)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        obs, _, terminated, truncated, info = self.env.step(1)
        if terminated or truncated:
            obs, info = self.env.reset(**kwargs)
        obs, _, terminated, truncated, info = self.env.step(2)
        if terminated or truncated:
            obs, info = self.env.reset(**kwargs)
        return obs, info


class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env=None, skip=4):
        super(MaxAndSkipEnv, self).__init__(env)
        self._obs_buffer = collections.deque(maxlen=2)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        info = {}
        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            if terminated or truncated:
                break
        max_frame = np.max(np.stack(self._obs_buffer), axis=0)
        return max_frame, total_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self._obs_buffer.clear()
        obs, info = self.env.reset(**kwargs)
        self._obs_buffer.append(obs)
        return obs, info


class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env=None):
        super(ProcessFrame84, self).__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)

    def observation(self, obs):
        return ProcessFrame84.process(obs)

    @staticmethod
    def process(frame):
        if frame.size == 210 * 160 * 3:
            img = np.reshape(frame, [210, 160, 3]).astype(np.float32)
        elif frame.size == 250 * 160 * 3:
            img = np.reshape(frame, [250, 160, 3]).astype(np.float32)
        else:
            assert False, "Unknown resolution."
        img = img[:, :, 0] * 0.299 + img[:, :, 1] * 0.587 + img[:, :, 2] * 0.114
        resized_screen = cv2.resize(img, (84, 110), interpolation=cv2.INTER_AREA)
        x_t = resized_screen[18:102, :]
        x_t = np.reshape(x_t, [84, 84, 1])
        return x_t.astype(np.uint8)


class ImageToPyTorch(gym.ObservationWrapper):
    def __init__(self, env):
        super(ImageToPyTorch, self).__init__(env)
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0,
            shape=(old_shape[-1], old_shape[0], old_shape[1]),
            dtype=np.float32
        )

    def observation(self, observation):
        return np.moveaxis(observation, 2, 0)


class ScaledFloatFrame(gym.ObservationWrapper):
    def observation(self, obs):
        return np.array(obs).astype(np.float32) / 255.0


class BufferWrapper(gym.ObservationWrapper):
    def __init__(self, env, n_steps, dtype=np.float32):
        super(BufferWrapper, self).__init__(env)
        self.dtype = dtype
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(
            old_space.low.repeat(n_steps, axis=0),
            old_space.high.repeat(n_steps, axis=0),
            dtype=dtype
        )

    def reset(self, **kwargs):
        self.buffer = np.zeros_like(self.observation_space.low, dtype=self.dtype)
        obs, info = self.env.reset(**kwargs)
        return self.observation(obs), info

    def observation(self, observation):
        self.buffer[:-1] = self.buffer[1:]
        self.buffer[-1] = observation
        return self.buffer


def make_env(env_name):
    env = gym.make(env_name, frameskip=1, repeat_action_probability=0.0)
    env = MaxAndSkipEnv(env)
    env = FireResetEnv(env)
    env = ProcessFrame84(env)
    env = ImageToPyTorch(env)
    env = BufferWrapper(env, 4)
    return ScaledFloatFrame(env)

## 4. Imports & Hyperparameters

In [ ]:
import time
import numpy as np
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from lib import wrappers

DEFAULT_ENV_NAME   = "ALE/Pong-v5"
MEAN_REWARD_BOUND  = 19.5
PER_MODEL_PATH     = "/kaggle/working/pong-per-noisy.dat"

# Standard DQN hyperparameters
GAMMA              = 0.99
BATCH_SIZE         = 32
REPLAY_SIZE        = 10000
LEARNING_RATE      = 1e-4
SYNC_TARGET_FRAMES = 1000
REPLAY_START_SIZE  = 10000

# PER hyperparameters
PER_ALPHA          = 0.6    # prioritization exponent (0=uniform, 1=full priority)
PER_BETA_START     = 0.4    # IS weight correction start (anneals to 1.0)
PER_BETA_FRAMES    = 100000 # frames to anneal beta over
PER_EPSILON        = 1e-6   # small constant so zero-error transitions keep non-zero priority

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 5. PER Data Structures — Sum Tree & Priority Buffer

In [ ]:
Experience = collections.namedtuple(
    'Experience',
    field_names=['state', 'action', 'reward', 'done', 'new_state']
)


class SumTree:
    """
    Binary tree where leaf nodes store priorities and internal nodes
    store the sum of their children. Gives O(log N) insert and sample.

    Layout (capacity=4):
                [0]          <- root: total priority sum
              /     \
           [1]       [2]     <- internal sums
          /   \     /   \
        [3]  [4] [5]  [6]   <- leaf priorities (experiences)
    """

    def __init__(self, capacity):
        self.capacity  = capacity
        self.tree      = np.zeros(2 * capacity - 1, dtype=np.float64)
        self.data      = np.empty(capacity, dtype=object)
        self.write_ptr = 0
        self.n_entries = 0

    def _propagate(self, leaf_idx, delta):
        """Propagate a priority change up the tree."""
        parent = (leaf_idx - 1) // 2
        self.tree[parent] += delta
        if parent != 0:
            self._propagate(parent, delta)

    def update(self, leaf_idx, priority):
        """Update priority of a leaf and propagate the change."""
        delta = priority - self.tree[leaf_idx]
        self.tree[leaf_idx] = priority
        self._propagate(leaf_idx, delta)

    def add(self, priority, data):
        """Add a new experience with given priority."""
        leaf_idx = self.write_ptr + self.capacity - 1
        self.data[self.write_ptr] = data
        self.update(leaf_idx, priority)
        self.write_ptr = (self.write_ptr + 1) % self.capacity
        self.n_entries = min(self.n_entries + 1, self.capacity)

    def _retrieve(self, node_idx, target):
        """Find leaf whose cumulative sum >= target."""
        left  = 2 * node_idx + 1
        right = left + 1
        if left >= len(self.tree):
            return node_idx           # reached a leaf
        if target <= self.tree[left]:
            return self._retrieve(left, target)
        else:
            return self._retrieve(right, target - self.tree[left])

    def get(self, target):
        """Sample one experience. Returns (leaf_idx, priority, experience)."""
        leaf_idx  = self._retrieve(0, target)
        data_idx  = leaf_idx - self.capacity + 1
        return leaf_idx, self.tree[leaf_idx], self.data[data_idx]

    @property
    def total(self):
        return self.tree[0]


class PERBuffer:
    """
    Prioritized Experience Replay buffer.
    Samples transitions proportional to their TD-error priority.
    Returns importance-sampling weights to correct for sampling bias.
    """

    def __init__(self, capacity, alpha=PER_ALPHA):
        self.alpha        = alpha
        self.tree         = SumTree(capacity)
        self.max_priority = 1.0      # new transitions get max priority so they are sampled at least once

    def __len__(self):
        return self.tree.n_entries

    def append(self, experience):
        """Add with maximum current priority so it gets sampled soon."""
        self.tree.add(self.max_priority ** self.alpha, experience)

    def sample(self, batch_size, beta):
        """
        Sample a batch using stratified sampling across priority segments.
        Returns batch + IS weights + tree indices (needed for priority updates).
        """
        batch      = []
        leaf_idxs  = []
        priorities = []
        segment    = self.tree.total / batch_size

        for i in range(batch_size):
            # Sample uniformly within each segment
            target = np.random.uniform(segment * i, segment * (i + 1))
            leaf_idx, priority, experience = self.tree.get(target)
            batch.append(experience)
            leaf_idxs.append(leaf_idx)
            priorities.append(priority)

        # Importance sampling weights — correct bias from non-uniform sampling
        probs   = np.array(priorities) / self.tree.total
        weights = (len(self) * probs) ** (-beta)
        weights /= weights.max()           # normalize so max weight = 1

        states, actions, rewards, dones, next_states = zip(*batch)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards,     dtype=np.float32),
            np.array(dones,       dtype=np.uint8),
            np.array(next_states),
            np.array(weights,     dtype=np.float32),
            leaf_idxs
        )

    def update_priorities(self, leaf_idxs, td_errors):
        """After a gradient step, refresh priorities with new TD errors."""
        for idx, td_error in zip(leaf_idxs, td_errors):
            priority = (abs(td_error) + PER_EPSILON) ** self.alpha
            self.tree.update(idx, priority)
            self.max_priority = max(self.max_priority, priority)


print("SumTree and PERBuffer defined.")

## 6. NoisyDQN Model & Agent

In [ ]:
class NoisyLinear(nn.Linear):
    """Linear layer with factorized Gaussian noise — replaces epsilon-greedy."""

    def __init__(self, in_features, out_features, sigma_init=0.5):
        super().__init__(in_features, out_features, bias=True)
        self.sigma_weight = nn.Parameter(
            torch.full((out_features, in_features), sigma_init / np.sqrt(in_features))
        )
        self.sigma_bias = nn.Parameter(
            torch.full((out_features,), sigma_init / np.sqrt(in_features))
        )
        self.register_buffer('epsilon_weight', torch.zeros(out_features, in_features))
        self.register_buffer('epsilon_bias',   torch.zeros(out_features))
        self.reset_noise()

    def reset_noise(self):
        eps_in  = self._scale_noise(self.in_features)
        eps_out = self._scale_noise(self.out_features)
        self.epsilon_weight.copy_(eps_out.outer(eps_in))
        self.epsilon_bias.copy_(eps_out)

    @staticmethod
    def _scale_noise(size):
        x = torch.randn(size)
        return x.sign() * x.abs().sqrt()

    def forward(self, x):
        if self.training:
            weight = self.weight + self.sigma_weight * self.epsilon_weight
            bias   = self.bias   + self.sigma_bias   * self.epsilon_bias
        else:
            weight = self.weight
            bias   = self.bias
        return F.linear(x, weight, bias)


class NoisyDQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),             nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),             nn.ReLU()
        )
        conv_out_size = self._get_conv_out(input_shape)
        self.fc = nn.Sequential(
            NoisyLinear(conv_out_size, 512), nn.ReLU(),
            NoisyLinear(512, n_actions)
        )

    def _get_conv_out(self, shape):
        o = self.conv(torch.zeros(1, *shape))
        return int(np.prod(o.size()))

    def forward(self, x):
        return self.fc(self.conv(x).view(x.size()[0], -1))

    def reset_noise(self):
        for layer in self.fc:
            if isinstance(layer, NoisyLinear):
                layer.reset_noise()


class NoisyAgent:
    def __init__(self, env, exp_buffer):
        self.env = env
        self.exp_buffer = exp_buffer
        self._reset()

    def _reset(self):
        self.state, _ = self.env.reset()
        self.total_reward = 0.0

    def play_step(self, net, device="cpu"):
        state_v = torch.tensor(np.asarray([self.state])).to(device)
        action  = int(torch.max(net(state_v), dim=1)[1].item())

        new_state, reward, terminated, truncated, _ = self.env.step(action)
        is_done = terminated or truncated
        self.total_reward += reward
        self.exp_buffer.append(
            Experience(self.state, action, reward, is_done, new_state)
        )
        self.state = new_state
        if is_done:
            done_reward = self.total_reward
            self._reset()
            return done_reward
        return None


def calc_loss_per(states, actions, rewards, dones, next_states, weights, net, tgt_net, device):
    """
    PER-aware loss: weights each sample's squared TD error by its IS weight.
    Also returns raw TD errors so we can update priorities in the buffer.
    """
    states_v      = torch.tensor(states).to(device)
    next_states_v = torch.tensor(next_states).to(device)
    actions_v     = torch.tensor(actions).to(device)
    rewards_v     = torch.tensor(rewards).to(device)
    done_mask     = torch.BoolTensor(dones).to(device)
    weights_v     = torch.tensor(weights).to(device)

    state_action_values = net(states_v).gather(1, actions_v.unsqueeze(-1)).squeeze(-1)
    next_state_values   = tgt_net(next_states_v).max(1)[0]
    next_state_values[done_mask] = 0.0
    expected = next_state_values.detach() * GAMMA + rewards_v

    td_errors = state_action_values - expected                        # raw errors for priority update
    loss      = (weights_v * td_errors.pow(2)).mean()                # IS-weighted MSE
    return loss, td_errors.detach().cpu().numpy()


print("NoisyDQN, NoisyAgent, calc_loss_per defined.")

## 7. Training — NoisyNets + PER

Key differences from plain NoisyNets:
- `ExperienceBuffer` → `PERBuffer` (priority-based sampling)
- `buffer.sample()` returns IS weights + leaf indices
- `calc_loss_per()` scales each sample's loss by its IS weight
- After each gradient step, priorities are refreshed with the new TD errors
- β anneals from 0.4 → 1.0 to gradually correct the sampling bias

In [ ]:
env     = wrappers.make_env(DEFAULT_ENV_NAME)
net     = NoisyDQN(env.observation_space.shape, env.action_space.n).to(device)
tgt_net = NoisyDQN(env.observation_space.shape, env.action_space.n).to(device)
print(net)

buffer    = PERBuffer(REPLAY_SIZE)
agent     = NoisyAgent(env, buffer)
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)

frame_idx        = 0
ts_frame         = 0
ts               = time.time()
total_rewards    = []
best_mean_reward = None

print("\nFilling replay buffer (10k frames) before training starts...")

while True:
    frame_idx += 1

    # Beta anneals from PER_BETA_START → 1.0 (full IS correction)
    beta = min(1.0, PER_BETA_START + frame_idx * (1.0 - PER_BETA_START) / PER_BETA_FRAMES)

    reward = agent.play_step(net, device=device)

    if reward is not None:
        total_rewards.append(reward)
        speed       = (frame_idx - ts_frame) / (time.time() - ts)
        ts_frame    = frame_idx
        ts          = time.time()
        mean_reward = np.mean(total_rewards[-100:])

        print("%d: done %d games, mean reward %.3f, beta %.3f, speed %.2f f/s" % (
            frame_idx, len(total_rewards), mean_reward, beta, speed
        ))

        if best_mean_reward is None or best_mean_reward < mean_reward:
            torch.save(net.state_dict(), PER_MODEL_PATH)
            if best_mean_reward is not None:
                print("  >> Best mean reward updated %.3f -> %.3f  [model saved]" % (
                    best_mean_reward, mean_reward
                ))
            best_mean_reward = mean_reward

        if mean_reward > MEAN_REWARD_BOUND:
            print("\nSolved in %d frames!" % frame_idx)
            break

    if len(buffer) < REPLAY_START_SIZE:
        continue

    if frame_idx % SYNC_TARGET_FRAMES == 0:
        tgt_net.load_state_dict(net.state_dict())

    # Resample noise once per training update
    net.reset_noise()
    tgt_net.reset_noise()

    # PER: sample with priorities, get IS weights and indices
    states, actions, rewards_b, dones, next_states, weights, leaf_idxs = \
        buffer.sample(BATCH_SIZE, beta)

    optimizer.zero_grad()
    loss, td_errors = calc_loss_per(
        states, actions, rewards_b, dones, next_states, weights,
        net, tgt_net, device
    )
    loss.backward()
    optimizer.step()

    # PER: update priorities with fresh TD errors
    buffer.update_priorities(leaf_idxs, td_errors)

env.close()

## 8. Results & Final Comparison

In [ ]:
import os

per_improvement    = (1 - frame_idx / 1_962_510) * 100
noisy_improvement  = (1 - 976_209   / 1_962_510) * 100

print("=" * 65)
print("FINAL COMPARISON — ALL ALGORITHMS")
print("=" * 65)
print(f"{'Algorithm':<25} {'Frames':>12} {'Games':>8} {'Mean Reward':>12} {'vs Baseline':>12}")
print("-" * 65)
print(f"{'Epsilon-Greedy':<25} {'1,962,510':>12} {'1073':>8} {'19.52':>12} {'baseline':>12}")
print(f"{'Boltzmann':<25} {'1,457,724':>12} {'687':>8} {'19.55':>12} {'+25.7%':>12}")
print(f"{'NoisyNets':<25} {'976,209':>12} {'559':>8} {'19.59':>12} {noisy_improvement:>11.1f}%")
print(f"{'NoisyNets + PER':<25} {frame_idx:>12,} {len(total_rewards):>8} {best_mean_reward:>12.3f} {per_improvement:>11.1f}%")
print("=" * 65)

if os.path.exists(PER_MODEL_PATH):
    size_mb = os.path.getsize(PER_MODEL_PATH) / (1024 * 1024)
    print(f"\nModel saved : {PER_MODEL_PATH}  ({size_mb:.2f} MB)")
    print("\nTo keep: Click 'Save Version' -> 'Save & Run All'")
else:
    print(f"\nWARNING: {PER_MODEL_PATH} not found.")

## 9. Play — Inference with NoisyNets + PER Model

In [ ]:
play_env = wrappers.make_env(DEFAULT_ENV_NAME)
play_net = NoisyDQN(play_env.observation_space.shape, play_env.action_space.n)
play_net.load_state_dict(
    torch.load(PER_MODEL_PATH, map_location=lambda storage, loc: storage)
)
play_net.eval()   # noise disabled during inference

state, _      = play_env.reset()
total_reward  = 0.0
action_counts = collections.Counter()

while True:
    state_v = torch.tensor(np.asarray([state]))
    action  = int(np.argmax(play_net(state_v).data.numpy()[0]))
    action_counts[action] += 1
    state, reward, terminated, truncated, _ = play_env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

play_env.close()
print("Total reward : %.2f" % total_reward)
print("Action counts:", dict(action_counts))